# 06 K 近邻 KNN

依赖安装说明：`pip install numpy matplotlib scikit-learn`

KNN 是一种非常直观的模型：预测一个新样本时，先找训练集中离它最近的 K 个样本，再用这些邻居投票或取平均。


## 1. 数学逻辑

常用欧氏距离：

$$d(x,z)=\sqrt{\sum_j(x_j-z_j)^2}$$

分类时，取距离最近的 K 个邻居：

$$N_K(x)=\text{K nearest samples to }x$$

再做多数投票：

$$\hat y=\text{mode}\{y_i|x_i\in N_K(x)\}$$

KNN 几乎没有显式训练，主要计算发生在预测阶段。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_moons(n_samples=250, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [ ]:
# 从零实现：KNN 分类

def knn_predict_one(x, X_train, y_train, k=5):
    distances = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
    neighbor_ids = np.argsort(distances)[:k]
    votes = Counter(y_train[neighbor_ids])
    return votes.most_common(1)[0][0]

def knn_predict(X_new, X_train, y_train, k=5):
    return np.array([knn_predict_one(x, X_train, y_train, k) for x in X_new])

for k in [1, 5, 21]:
    pred = knn_predict(X_test_s, X_train_s, y_train, k=k)
    print(f'k={k:2d} | accuracy={accuracy_score(y_test, pred):.3f}')


In [ ]:
# sklearn 实战：KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
print('sklearn accuracy:', round(accuracy_score(y_test, pred), 3))

xx, yy = np.meshgrid(np.linspace(X_train_s[:,0].min()-1, X_train_s[:,0].max()+1, 180),
                     np.linspace(X_train_s[:,1].min()-1, X_train_s[:,1].max()+1, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=25)
plt.title('KNN 决策边界')
plt.show()


## 2. 常见误区

- KNN 对特征尺度极其敏感，通常必须标准化。
- K 太小容易过拟合，K 太大容易欠拟合。
- 高维空间中距离会变得不可靠，这叫维度灾难。

## 3. 小实验

- 改 `k`，观察边界平滑程度。
- 去掉标准化，观察效果变化。
- 把 `noise` 调大，看 KNN 何时变得不稳定。
